In [1]:
import numpy as np
from scipy.stats import t
from time import time
from abc import ABC
import warnings

In [2]:
"""
N: Number of choice situations
P: Number of observations per panel
J: Number of alternatives
K: Number of variables (Kf: fixed, Kr: random)
"""

'\nN: Number of choice situations\nP: Number of observations per panel\nJ: Number of alternatives\nK: Number of variables (Kf: fixed, Kr: random)\n'

In [ ]:
class ChoiceModel(ABC):
    """
    Base class for estimation of discrete choice models
    """
    def __init__(self):
        self.coeff_names = None
        self.coeff_ = None
        self.stderr = None
        self.zvalues = None
        self.pvalues = None
        self.loglikelihood = None
        self.total_fun_eval = 0
        self.verbose = 1
        self.robust = False
    
    def _reset_attributes(self):
        self.coeff_names = None
        self.coeff_ = None
        self.stderr = None
        self.zvalues = None
        self.pvalues = None
        self.loglikelihood = None
        self.total_fun_eval = 0
        self.verbose = 1
        self.robust = False
        
    def _as_array(self, X, y, varnames, alts, isvars, ids, weights, panels, 
                  avail, scale_factor):
        X = np.asarray(X)
        y = np.asarray(y)
        varnames = np.asarray(varnames) if varnames is not None else None
        alts = np.asarray(alts) if alts is not None else None
        isvars = np.asarray(isvars) if isvars is not None else None
        ids = np.asarray(ids) if ids is not None else None
        weights = np.asarray(weights) if weights is not None else None
        panels = np.asarray(panels) if panels is not None else None
        avail = np.asarray(avail) if avail is not None else None
        scale_factor = np.asarray(scale_factor) if scale_factor is not None else None
        return X, y, varnames, alts, isvars, ids, weights, panels, avail, scale_factor
    
    def _pre_fit(self, alts, varnames, isvars, base_alt, fit_intercept, 
                 maxiter):
        self._reset_attributes()
        self._fit_start_time = time()
        self._isvars = [] if isvars is None else list(isvars)
        self.asvars = [v for v in varnames if v not in self._isvars]
        self._varnames = list(varnames)
        self._fit_intercept = fit_intercept
        self.alternatives = np.sort(np.unique(alts))
        self.base_alt = self.alternatives[0] if base_alt is None else base_alt
        self.maxiter = maxiter
        
    def _post_fit(self, optim_res, coeff_names, sample_size, verbose=1, robust=False):
        self.convergence = optim_res['success']
        self.coeff_ = optim_res['x']
        self.hess_inv = optim_res['hess_inv']
        self.covariance = self._robust_covariance(optim_res['hess_inv'], optim_res['grad_n']) \
            if robust else optim_res['hess_inv']
        self.stderr = np.sqrt(np.diag(self.covariance))
        self.zvalues = self.coeff_ / self.stderr
        self.pvalues = 2 * (1 - t.cdf(np.abs(self.zvalues), df=sample_size))
        self.loglikelihood = -optim_res['fun']
        self.estimation_message = optim_res['message']
        self.coeff_names = coeff_names
        self.total_iter = optim_res['nit']
        self.estim_time_sec = time() - self._fit_start_time
        self.sample_size = sample_size
        self.aic = 2*len(self.coeff_) - 2*self.loglikelihood
        self.bic = np.log(sample_size) * len(self.coeff_) - 2*self.loglikelihood
        self.grad_n = optim_res['grad_n']
        self.total_fun_eval = optim_res['nfev']
        
        if not self.convergence and verbose > 0:
            print("**** The optimization did not converge after {}"
                  " iterations. ****".format(self.total_iter))
            print("Message: {}".format(self.estimation_message))
    
    def _robust_covariance(self, hess_inv, grad_n):
        """
        Compute the robust covariance matrix
        """
        pass